In [5]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw")

PP_COLUMNS = [
    "transaction_id", "price", "date", "postcode",
    "property_type", "old_new", "duration", "paon",
    "saon", "street", "locality", "town",
    "district", "county", "ppd_category", "record_status"
]

FILES = [
    "pp-2023.csv",
    "pp-2024.csv",
    "pp-2025.csv",
]

def load_price_paid(filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(
        path,
        header=None,
        names=PP_COLUMNS,
        dtype={
            "transaction_id": "string",
            "postcode": "string",
            "paon": "string",
            "saon": "string",
        },
        low_memory=False
    )

    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    print(f"\n--- {filename} ---")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Date range:", df["date"].min(), "to", df["date"].max())
    print("Missing prices:", df["price"].isna().sum())
    print("Duplicate transaction IDs:", df["transaction_id"].duplicated().sum())

    return df

yearly_data = [load_price_paid(f) for f in FILES]
combined = pd.concat(yearly_data, ignore_index=True)

print("\n--- COMBINED DATA ---")
print("Total rows:", len(combined))
print("Unique transaction IDs:", combined["transaction_id"].nunique())
print("\nProperty types:")
print(combined["property_type"].value_counts(dropna=False))
print("\nTop 15 counties:")
print(combined["county"].value_counts().head(15))

lookup_path = DATA_DIR / "pp-uprn-lookup-aug-2026.csv"

if lookup_path.exists():
    lookup = pd.read_csv(
        lookup_path,
        header=None,
        dtype="string",
        keep_default_na=False,
        low_memory=False
    )

    print("\n--- UPRN LOOKUP ---")
    print("Shape:", lookup.shape)
    print("First rows:")
    print(lookup.head(5).to_string(index=False))
    print("\nNon-empty values by column:")
    print((lookup != "").sum())

    # The expected layout is transaction ID in column 1,
    # UPRN in column 2. Verify the printed sample before joining.
    if lookup.shape[1] >= 2:
        print("\nLookup sample IDs:")
        print(lookup.iloc[:5, 0].tolist())
        print("Lookup sample UPRNs:")
        print(lookup.iloc[:5, 1].tolist())
else:
    print(f"\nLookup file not found: {lookup_path}")

print("\nInspection complete. No source files were modified.")
    



--- pp-2023.csv ---
Rows: 861502
Columns: 16
Date range: 2023-01-01 00:00:00 to 2023-12-31 00:00:00
Missing prices: 0
Duplicate transaction IDs: 0

--- pp-2024.csv ---
Rows: 931453
Columns: 16
Date range: 2024-01-01 00:00:00 to 2024-12-31 00:00:00
Missing prices: 0
Duplicate transaction IDs: 0

--- pp-2025.csv ---
Rows: 974087
Columns: 16
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00
Missing prices: 0
Duplicate transaction IDs: 0

--- COMBINED DATA ---
Total rows: 2767042
Unique transaction IDs: 2767042

Property types:
property_type
T    750368
S    749128
D    624874
F    486636
O    156036
Name: count, dtype: int64

Top 15 counties:
county
GREATER LONDON        311469
GREATER MANCHESTER    128670
WEST YORKSHIRE        107785
WEST MIDLANDS         103129
KENT                   77333
ESSEX                  74232
LANCASHIRE             69238
MERSEYSIDE             67156
HAMPSHIRE              66402
SOUTH YORKSHIRE        64410
SURREY                 56015
TYNE AND WEAR       